# Proyecto Curso Intermedio de Analítica Avanzada

En este archivo se encuentran las pruebas realizadas para validar los modelos con sus parámetros optimizados. Este repositorio ha sido elaborado para poder analizar ciertas problemáticas de la NBA con modelos de machine learning y deep learning. Los autores de este proyecto son:

- Luis Alberto Arias Llaguno
- David Ceballos Mata
- Rubén Octavio Flores Ramos

En este archivo se verá problema por problema los resultados obtenidos para validar los modelos con sus parámetros optimizados. Asimismo, se verá la exploración de datos y la limpieza de datos de la api de la NBA para poder obtener los datos necesarios para los modelos.

# 0. Importación de librerías

In [1]:
# Instalación de librerías

%pip install nba_api
%pip install tensorflow
%pip install keras
%pip install keras_tuner

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importación de librerías

from nba_api.stats.endpoints import leaguegamefinder
from nba_api.stats.static import teams
from nba_api.stats.static import players
from nba_api.stats.endpoints import leaguedashplayerstats

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay, silhouette_score, r2_score
from sklearn.cluster import KMeans
from sklearn.neighbors import KNeighborsClassifier

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
import keras_tuner as kt

import mlflow
import mlflow.tensorflow

C:\Users\luis_\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\luis_\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\luis_\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Pyt

# 1. Exploración de datos

# 2. Problemas a Resolver

## 2.1 Predicción de rendimiento partido a partido usando LSTM (Parte 1) (Luis)

### Problema: 
Predecir la línea de puntos, de un equipo (en este caso Lakers) en su próximo partido tomando en cuenta los juegos anteriores. Para este problema, vamos a probar con los 3, 5, y 10 juegos anteriores para poder predecir los puntos de los futuros juegos de los lakers. En esta parte 1 se puede ver que solo se toman en cuenta los juegos de los Lakers como características para el modelo de redes neuronales.

### 2.1.1 Obtención y de datos

In [41]:
# Obtener el ID de los Lakers
nba_teams = teams.get_teams()
lakers = [team for team in nba_teams if team['full_name'] == 'Los Angeles Lakers'][0]
lakers_id = lakers['id']

# Usar LeagueGameFinder para obtener todos los juegos de los Lakers
gamefinder = leaguegamefinder.LeagueGameFinder(team_id_nullable=lakers_id)
games = gamefinder.get_data_frames()[0]

# Crear DataFrame con las columnas más relevantes
df_lakers = games[[
    'SEASON_ID',           # Temporada
    'GAME_ID',             # ID del juego
    'GAME_DATE',           # Fecha del juego
    'MATCHUP',             # Matchup (LAL vs. OPP o LAL @ OPP)
    'WL',                  # Win/Loss
    'MIN',                 # Minutos jugados
    'PTS',                 # Puntos anotados
    'FGM',                 # Field Goals Made (tiros de campo anotados)
    'FGA',                 # Field Goals Attempted (tiros de campo intentados)
    'FG_PCT',              # Porcentaje de tiros de campo
    'FG3M',                # 3-Point Field Goals Made (triples anotados)
    'FG3A',                # 3-Point Field Goals Attempted (triples intentados)
    'FG3_PCT',             # Porcentaje de triples
    'FTM',                 # Free Throws Made (tiros libres anotados)
    'FTA',                 # Free Throws Attempted (tiros libres intentados)
    'FT_PCT',              # Porcentaje de tiros libres
    'OREB',                # Rebotes ofensivos
    'DREB',                # Rebotes defensivos
    'REB',                 # Rebotes totales
    'AST',                 # Asistencias
    'STL',                 # Robos
    'BLK',                 # Bloqueos
    'TOV',                 # Pérdidas de balón
    'PF',                  # Faltas personales
    'PLUS_MINUS'           # Plus/Minus
]].copy()

df_lakers.info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4101 entries, 0 to 4100
Data columns (total 25 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   SEASON_ID   4101 non-null   object 
 1   GAME_ID     4101 non-null   object 
 2   GAME_DATE   4101 non-null   object 
 3   MATCHUP     4101 non-null   object 
 4   WL          4099 non-null   object 
 5   MIN         4101 non-null   int64  
 6   PTS         4101 non-null   int64  
 7   FGM         4101 non-null   int64  
 8   FGA         4101 non-null   int64  
 9   FG_PCT      4101 non-null   float64
 10  FG3M        4101 non-null   int64  
 11  FG3A        4101 non-null   int64  
 12  FG3_PCT     4089 non-null   float64
 13  FTM         4101 non-null   int64  
 14  FTA         4101 non-null   int64  
 15  FT_PCT      4099 non-null   float64
 16  OREB        4101 non-null   int64  
 17  DREB        4101 non-null   int64  
 18  REB         4101 non-null   int64  
 19  AST         4101 non-null  

### 2.1.2 Limpieza de datos

In [42]:
# Convertir la fecha a formato datetime
df_lakers['GAME_DATE'] = pd.to_datetime(df_lakers['GAME_DATE']) 

# Ordenar por fecha (más reciente primero)
df_lakers = df_lakers.sort_values('GAME_DATE', ascending=True).reset_index(drop=True) 

# Agregar columna para identificar si es juego en casa o de visitante
df_lakers['ES_CASA'] = df_lakers['MATCHUP'].str.contains('vs.').astype(int)

# Extraer el oponente
df_lakers['OPONENTE'] = df_lakers['MATCHUP'].str.split().str[-1]

# Eliminar plus_minus
df_lakers = df_lakers.drop(columns=['PLUS_MINUS']) 

# Eliminar filas con ANY valor nulo en df_lakers
df_lakers = df_lakers.dropna().reset_index(drop=True) 

# Crear columna WIN (1 para victoria, 0 para derrota)
df_lakers['WIN'] = (df_lakers['WL'] == 'W').astype(int)

# Contar los días que hubo entre juegos
df_lakers['DAYS_REST'] = df_lakers['GAME_DATE'].diff().dt.days
df_lakers['DAYS_REST'] = df_lakers['DAYS_REST'].fillna(df_lakers['DAYS_REST'].median()) # Rellenar valores nulos con la mediana
df_lakers['DAYS_REST'] = df_lakers['DAYS_REST'].astype(int)

# Visualizar los datos
pd.set_option('display.max_columns', None)
df_lakers.info()
df_lakers.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4087 entries, 0 to 4086
Data columns (total 28 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   SEASON_ID  4087 non-null   object        
 1   GAME_ID    4087 non-null   object        
 2   GAME_DATE  4087 non-null   datetime64[ns]
 3   MATCHUP    4087 non-null   object        
 4   WL         4087 non-null   object        
 5   MIN        4087 non-null   int64         
 6   PTS        4087 non-null   int64         
 7   FGM        4087 non-null   int64         
 8   FGA        4087 non-null   int64         
 9   FG_PCT     4087 non-null   float64       
 10  FG3M       4087 non-null   int64         
 11  FG3A       4087 non-null   int64         
 12  FG3_PCT    4087 non-null   float64       
 13  FTM        4087 non-null   int64         
 14  FTA        4087 non-null   int64         
 15  FT_PCT     4087 non-null   float64       
 16  OREB       4087 non-null   int64         


,SEASON_ID,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,ES_CASA,OPONENTE,WIN,DAYS_REST
0,21983,0028300010,1983-10-29,LAL @ UTH,W,240,120,46,98,0.469,0,1,0.000,28,39,0.718,29,31,60,25,10,10,20,41,0,UTH,1,2
1,21983,0028300035,1983-11-02,LAL @ SDC,L,240,106,43,96,0.448,1,2,0.500,19,25,0.760,16,33,49,29,14,7,24,24,0,SDC,0,4
2,21983,0028300053,1983-11-05,LAL @ DAL,L,240,102,47,97,0.485,0,2,0.000,8,12,0.667,18,23,41,29,7,8,20,27,0,DAL,0,3
3,21983,0028300066,1983-11-08,LAL @ DEN,W,240,133,49,106,0.462,2,3,0.667,33,36,0.917,22,31,53,30,11,8,23,31,0,DEN,1,3
4,21983,0028300068,1983-11-09,LAL vs. DAL,W,240,120,50,94,0.532,2,3,0.667,18,28,0.643,16,33,49,33,7,6,17,31,1,DAL,1,1


### 2.1.3 Definición de columnas de features y target y Creación de Datasets de Entrenamiento y Validación

In [43]:
feature_cols = [
    'ES_CASA',
    'WIN',
    'MIN',
    'FGM', 'FGA', 'FG_PCT',
    'FG3M', 'FG3A', 'FG3_PCT',
    'FTM', 'FTA', 'FT_PCT',
    'OREB', 'DREB', 'REB',
    'AST', 'STL', 'BLK',
    'TOV', 'PF',
    'DAYS_REST'
]

target_col = 'PTS'

data_features = df_lakers[feature_cols].values.astype('float32')
data_target = df_lakers[target_col].values.astype('float32')

print("Features shape (sin secuencias):", data_features.shape)
print("Target shape (sin secuencias):", data_target.shape)

Features shape (sin secuencias): (4087, 21)
Target shape (sin secuencias): (4087,)


In [44]:
# Función para construir secuencias de longitud N
def build_sequences(features, target, seq_len=5):
    X, y = [], []
    n_samples = len(features)
    
    for i in range(n_samples - seq_len):
        seq_x = features[i:i+seq_len]
        seq_y = target[i+seq_len]    # Puntos del partido siguiente
        X.append(seq_x)
        y.append(seq_y)
    
    return np.array(X), np.array(y)

# Función para preparar datos para secuencias de longitud N
def prepare_data_for_seq_len(seq_len):
    X, y = build_sequences(data_features, data_target, seq_len=seq_len)
    print(f"\nSEQ_LEN = {seq_len} -> X: {X.shape}, y: {y.shape}")
    
    # split temporal 70/15/15
    n = X.shape[0]
    train_size = int(n * 0.7)
    val_size   = int(n * 0.15)
    
    X_train = X[:train_size]
    y_train = y[:train_size]
    X_val   = X[train_size:train_size+val_size]
    y_val   = y[train_size:train_size+val_size]
    X_test  = X[train_size+val_size:]
    y_test  = y[train_size+val_size:]
    
    num_features = X.shape[2]
    
    # Escalar
    X_train_2d = X_train.reshape(-1, num_features)
    X_val_2d   = X_val.reshape(-1, num_features)
    X_test_2d  = X_test.reshape(-1, num_features)
    
    scaler = StandardScaler()
    X_train_scaled_2d = scaler.fit_transform(X_train_2d)
    X_val_scaled_2d   = scaler.transform(X_val_2d)
    X_test_scaled_2d  = scaler.transform(X_test_2d)
    
    X_train_scaled = X_train_scaled_2d.reshape(X_train.shape)
    X_val_scaled   = X_val_scaled_2d.reshape(X_val.shape)
    X_test_scaled  = X_test_scaled_2d.reshape(X_test.shape)
    
    return (
        X_train_scaled, y_train,
        X_val_scaled, y_val,
        X_test_scaled, y_test,
        num_features, scaler
    )

### 2.1.4 Modelo Tuneado con Keras

In [45]:
import tensorflow as tf
from tensorflow.keras import layers, models
import keras_tuner as kt

def build_lstm_model(hp, seq_len, num_features):
    model = tf.keras.Sequential()
    
    units = hp.Int('lstm_units', min_value=16, max_value=128, step=16)
    model.add(
        layers.LSTM(
            units,
            return_sequences=False,
            input_shape=(seq_len, num_features)
        )
    )
    
    dropout_rate = hp.Float('dropout_rate', 0.0, 0.5, step=0.1)
    model.add(layers.Dropout(dropout_rate))
    
    dense_units = hp.Int('dense_units', min_value=8, max_value=64, step=8)
    model.add(layers.Dense(dense_units, activation='relu'))
    
    model.add(layers.Dense(1, activation='linear'))
    
    optimizer_name = hp.Choice('optimizer', ['sgd', 'rmsprop', 'adam'])
    learning_rate = hp.Float(
        'learning_rate',
        min_value=1e-4,
        max_value=1e-2,
        sampling='log'
    )
    
    clipnorm = hp.Choice('clipnorm', values=[0.0, 1.0, 2.0])
    clip_value = None if clipnorm == 0.0 else clipnorm
    
    if optimizer_name == 'sgd':
        momentum = hp.Float('momentum', min_value=0.0, max_value=0.9, step=0.3)
        optimizer = tf.keras.optimizers.SGD(
            learning_rate=learning_rate,
            momentum=momentum,
            clipnorm=clip_value
        )
    elif optimizer_name == 'rmsprop':
        optimizer = tf.keras.optimizers.RMSprop(
            learning_rate=learning_rate,
            clipnorm=clip_value
        )
    else:
        optimizer = tf.keras.optimizers.Adam(
            learning_rate=learning_rate,
            clipnorm=clip_value
        )
    
    model.compile(
        optimizer=optimizer,
        loss='mse',
        metrics=['mae']
    )
    
    return model


class BatchSizeTuner(kt.Hyperband):
    """Tuner que también tunea batch_size vía hp."""
    def run_trial(self, trial, *args, **kwargs):
        hp = trial.hyperparameters
        batch_size = hp.Choice('batch_size', [16, 32, 64])
        kwargs['batch_size'] = batch_size
        return super().run_trial(trial, *args, **kwargs)


### 2.1.5 MLflow con la búsqueda de hiperparámetros óptimos

In [8]:
import os
import shutil
from pathlib import Path
import mlflow
import mlflow.tensorflow

# 1. Carpeta raíz PARA MLflow (fuera del repo)
ml_root = Path(r"C:\Downloads\mlflow_nba")

# 2. Borrar lo que hubiera antes (si quieres empezar limpio)
shutil.rmtree(ml_root, ignore_errors=True)

# 3. Crear la carpeta raíz y la subcarpeta .trash (MUY IMPORTANTE)
ml_root.mkdir(parents=True, exist_ok=True)
trash_dir = ml_root / ".trash"
trash_dir.mkdir(exist_ok=True)

print("MLflow root:", ml_root)
print("Existe root:", ml_root.exists(), " / es dir:", ml_root.is_dir())
print("Existe .trash:", trash_dir.exists(), " / es dir:", trash_dir.is_dir())

# 4. Configurar tracking_uri para que apunte ahí
tracking_uri = ml_root.as_uri()  # file:///C:/Downloads/mlflow_nba
mlflow.set_tracking_uri(tracking_uri)
print("Tracking URI actual:", mlflow.get_tracking_uri())

# 5. Crear/usar experimento y activar autolog
mlflow.set_experiment("nba_lakers_lstm")
mlflow.tensorflow.autolog(disable=True)


C:\Users\luis_\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\mlflow\tracking\_tracking_service\utils.py:140: FutureWarning: Filesystem tracking backend (e.g., './mlruns') is deprecated. Please switch to a database backend (e.g., 'sqlite:///mlflow.db'). For feedback, see: https://github.com/mlflow/mlflow/issues/18534
  return FileStore(store_uri, store_uri)
2025/12/11 13:55:28 INFO mlflow.tracking.fluent: Experiment with name 'nba_lakers_lstm' does not exist. Creating a new experiment.


MLflow root: C:\Downloads\mlflow_nba
Existe root: True  / es dir: True
Existe .trash: True  / es dir: True
Tracking URI actual: file:///C:/Downloads/mlflow_nba


In [9]:
with mlflow.start_run(run_name="prueba_en_downloads"):
    mlflow.log_param("test_param", 123)
    mlflow.log_metric("test_metric", 0.99)

print("Contenido de C:\\Downloads\\mlflow_nba:", os.listdir(ml_root))
for name in os.listdir(ml_root):
    print("  ", name)


Contenido de C:\Downloads\mlflow_nba: ['.trash', '511604131770777317']
   .trash
   511604131770777317


In [10]:
from pathlib import Path
import shutil

kt_dir = Path(r"E:\001 - Portafolio Anáhuac\01 - Ingeniería en Tecnologías de la Información\4to Semestre\Analítica Avanzada\proyecto-analitica-nba") / "kt_lakers_runs"
shutil.rmtree(kt_dir, ignore_errors=True)
kt_dir.mkdir(parents=True, exist_ok=True)
base_dir = kt_dir.as_posix()

print("Keras Tuner dir:", base_dir)


Keras Tuner dir: E:/001 - Portafolio Anáhuac/01 - Ingeniería en Tecnologías de la Información/4to Semestre/Analítica Avanzada/proyecto-analitica-nba/kt_lakers_runs


In [11]:
import os
import shutil
from pathlib import Path

# Carpeta para Keras Tuner (FUERA del repo)
kt_root = Path(r"C:\Downloads\kt_lakers_lstm")

# Borramos cualquier cosa vieja
shutil.rmtree(kt_root, ignore_errors=True)

# Creamos carpeta limpia
kt_root.mkdir(parents=True, exist_ok=True)

# Usamos ruta tipo POSIX (con /) que TF maneja bien
base_dir = kt_root.as_posix()

print("Keras Tuner base_dir:", base_dir)
print("Existe:", os.path.exists(base_dir), " / es dir:", os.path.isdir(base_dir))
print("Contenido:", os.listdir(base_dir))


Keras Tuner base_dir: C:/Downloads/kt_lakers_lstm
Existe: True  / es dir: True
Contenido: []


In [12]:
seq_len_candidates = [3, 5, 10]   # SEQ_LEN (los juegos anteriores que se requieren como contexto para predecir el siguiente)
max_epochs = 40

results = []

for seq_len in seq_len_candidates:
    (
        X_train_scaled, y_train,
        X_val_scaled, y_val,
        X_test_scaled, y_test,
        num_features, scaler
    ) = prepare_data_for_seq_len(seq_len)

    with mlflow.start_run(run_name=f"tuning_seq{seq_len}"):

        mlflow.log_param("seq_len", seq_len)

        def hypermodel(hp):
            return build_lstm_model(hp, seq_len, num_features)

        tuner = BatchSizeTuner(
            hypermodel,
            objective='val_loss',
            max_epochs=max_epochs,
            factor=3,
            directory=base_dir,
            project_name=f"seq_{seq_len}",
            overwrite=True
        )

        stop_early = EarlyStopping(
            monitor='val_loss',
            patience=5,
            restore_best_weights=True
        )

        tuner.search(
            X_train_scaled, y_train,
            validation_data=(X_val_scaled, y_val),
            callbacks=[stop_early],
            verbose=1
        )

        # Mejor conjunto de hiperparámetros
        best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]
        best_trial = tuner.oracle.get_best_trials(num_trials=1)[0]
        best_val_loss = best_trial.score

        # Construir modelo ganador
        best_model = tuner.hypermodel.build(best_hp)

        history = best_model.fit(
            np.concatenate([X_train_scaled, X_val_scaled], axis=0),
            np.concatenate([y_train, y_val], axis=0),
            epochs=max_epochs,
            batch_size=best_hp.get('batch_size'),
            callbacks=[stop_early],
            verbose=0
        )

        test_loss, test_mae = best_model.evaluate(X_test_scaled, y_test, verbose=0)

        # Logear solo hiperparámetros ganadores y métricas finales
        mlflow.log_metric("best_val_loss", float(best_val_loss))
        mlflow.log_metric("test_mse", float(test_loss))
        mlflow.log_metric("test_mae", float(test_mae))

        for name, value in best_hp.values.items():
            # para evitar conflictos con nombres reservados, prefijamos
            mlflow.log_param(f"best_{name}", value)

        # Guardar en lista para imprimir resumen al final
        run_result = {
            "seq_len": seq_len,
            "best_val_loss": float(best_val_loss),
            "test_mse": float(test_loss),
            "test_mae": float(test_mae),
        }
        for name, value in best_hp.values.items():
            run_result[name] = value

        results.append(run_result)

Trial 90 Complete [00h 00m 14s]
val_loss: 184.5052490234375

Best val_loss So Far: 158.71412658691406
Total elapsed time: 00h 09m 33s


C:\Users\luis_\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\keras\src\callbacks\early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss,mae
  current = self.get_monitor_value(logs)


### 2.1.6 Métricas e Interpretación

In [13]:
def print_results_for_seq_len(results, seq_len):
    print("\n" + "="*80)
    print(f"======================= RESULTADOS COMPLETOS: SEQ_LEN = {seq_len} =======================")
    print("="*80)

    filtered = [r for r in results if r["seq_len"] == seq_len]
    if not filtered:
        print(f"No hay resultados para SEQ_LEN={seq_len}.")
        return

    r = filtered[0]

    # Métricas
    print(f"\nbest_val_loss : {r['best_val_loss']}")
    print(f"test_mse       : {r['test_mse']}")
    print(f"test_mae       : {r['test_mae']}")

    print("\nHiperparámetros ganadores:")
    for k, v in r.items():
        if k not in ["seq_len", "best_val_loss", "test_mse", "test_mae"]:
            print(f"  - {k}: {v}")

    print("="*80 + "\n")


# Imprimir resultados completos para cada SEQ_LEN
print_results_for_seq_len(results, 3)


======================= RESULTADOS COMPLETOS: SEQ_LEN = 3 =======================

best_val_loss : 165.90769958496094
test_mse       : 205.82420349121094
test_mae       : 11.53972053527832

Hiperparámetros ganadores:
  - lstm_units: 16
  - dropout_rate: 0.4
  - dense_units: 8
  - optimizer: sgd
  - learning_rate: 0.0038011372648135925
  - clipnorm: 1.0
  - momentum: 0.8999999999999999
  - batch_size: 32
  - tuner/epochs: 40
  - tuner/initial_epoch: 0
  - tuner/bracket: 0
  - tuner/round: 0



In [14]:
print_results_for_seq_len(results, 5)


======================= RESULTADOS COMPLETOS: SEQ_LEN = 5 =======================

best_val_loss : 161.1466827392578
test_mse       : 230.9376220703125
test_mae       : 12.219761848449707

Hiperparámetros ganadores:
  - lstm_units: 128
  - dropout_rate: 0.30000000000000004
  - dense_units: 16
  - optimizer: sgd
  - learning_rate: 0.0017311246306854196
  - clipnorm: 0.0
  - momentum: 0.0
  - batch_size: 64
  - tuner/epochs: 40
  - tuner/initial_epoch: 14
  - tuner/bracket: 2
  - tuner/round: 2
  - tuner/trial_id: 0064



In [15]:
print_results_for_seq_len(results, 10)


======================= RESULTADOS COMPLETOS: SEQ_LEN = 10 =======================

best_val_loss : 158.71412658691406
test_mse       : 307.41864013671875
test_mae       : 13.860396385192871

Hiperparámetros ganadores:
  - lstm_units: 48
  - dropout_rate: 0.0
  - dense_units: 24
  - optimizer: adam
  - learning_rate: 0.00306562330900601
  - clipnorm: 1.0
  - momentum: 0.6
  - batch_size: 32
  - tuner/epochs: 40
  - tuner/initial_epoch: 14
  - tuner/bracket: 3
  - tuner/round: 3
  - tuner/trial_id: 0048



En este caso podemos ver que el modelo que usa los 10 juegos anteriores para predecir el siguiente juego es el que tiene mejores métricas. Esto debido a la métrica de MAE (Mean Absolute Error), la cual con su valor de 11.3334 nos dice que el modelo promedio de error es de 11.3334 puntos. En otras palabras, esto dice que el modelo está prediciendo con un error promedio de 11.3334 puntos, osea que de su predicción está en promedio 11.3334 puntos de distancia de la realidad.

In [16]:
# R^2 score

y_pred = best_model.predict(X_test_scaled).flatten()   # predicciones
r2 = r2_score(y_test, y_pred)

print("Coeficiente de determinación R²:", r2)

20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step
Coeficiente de determinación R²: -0.5687004327774048


In [17]:
df_lakers['PTS'].mean()

105.26620993393688

Aquí hay que hacer un detenimiento especial en la evaluación de la métrica de R², porque al ser negativo teóricamente significa que el modelo mejora que si es que simplemente sacara el promedio de los valores de la variable objetivo para los juegos anteriores y predijeramos con eso. Esto se debe a un par de factores, principalmente que el promedio de puntos de todos los juegos (que hay en nba_api) de los Lakers es de 105.26, por lo que los juegos de los lakers son bastante estables. Otro factor que tomar en cuenta es que no hay características de los contrincantes, por lo que el modelo no tiene información sobre quién es el contrincante en el juego actual y esto también llega a afectar la precisión de la predicción.

## 2.2 Predicción de rendimiento partido a partido usando LSTM (Parte 2) (Luis)

### Problema: 
Predecir la línea de puntos, de un equipo (en este caso Lakers) en su próximo partido tomando en cuenta los juegos anteriores. Para este problema, vamos a probar con los 3, 5, y 10 juegos anteriores para poder predecir los puntos de los futuros juegos de los lakers. En esta parte 2 se puede ver que no solo se toman en cuenta las características de los Lakers en ese juego, sino que también se toman en cuenta las características de los contrincantes en el mismo juego como características para el modelo de redes neuronales. En teoría, estas nuevas características beneficiarían al modelo debido a que tener las características del contrincante en ese partido también da más contexto sobre el juego y de los contrincantes como han sido en el pasado para poder predecir con mayor exactitud el rendimiento de los Lakers en el próximo partido.

### 2.2.1 Crear dataframe con stats de los contrincantes y Lakers

In [46]:
teams_df = pd.DataFrame(nba_teams)
abbr_to_id = dict(zip(teams_df["abbreviation"], teams_df["id"]))

opp_abbrevs = df_lakers["OPONENTE"].unique()
print("Oponentes únicos:", opp_abbrevs)

# ¿Qué abreviaturas NO están en los equipos oficiales?
missing = [abbr for abbr in opp_abbrevs if abbr not in abbr_to_id]
print("Abreviaturas que no están en nba_teams:", missing)


Oponentes únicos: ['UTH' 'SDC' 'DAL' 'DEN' 'PHX' 'MIL' 'CLE' 'POR' 'SEA' 'CHI' 'GOS' 'NYK'
 'SAN' 'HOU' 'KCK' 'IND' 'BOS' 'WAS' 'ATL' 'PHL' 'NJN' 'DET' 'LAC' 'SAC'
 'MIA' 'CHH' 'ORL' 'MIN' 'VAN' 'TOR' 'SAS' 'GSW' 'UTA' 'PHI' 'MEM' 'NOH'
 'CHA' 'NOK' 'BAR' 'OKC' 'BKN' 'NOP' 'MAC' 'SKT']
Abreviaturas que no están en nba_teams: ['UTH', 'SDC', 'SEA', 'GOS', 'SAN', 'KCK', 'PHL', 'NJN', 'CHH', 'VAN', 'NOH', 'NOK', 'BAR', 'MAC', 'SKT']


In [47]:
# Mapa manual de abreviaturas raras a abreviaturas oficiales de nba_api
abbr_fix_map = {
    "UTH": "UTA",  # Utah Jazz
    "SDC": "LAC",  # San Diego Clippers -> LA Clippers
    "SEA": "OKC",  # Seattle SuperSonics -> Oklahoma City Thunder
    "GOS": "GSW",  # Golden State (a veces codificado raro) -> Golden State Warriors
    "SAN": "SAS",  # San Antonio -> San Antonio Spurs
    "KCK": "SAC",  # Kansas City Kings -> Sacramento Kings
    "PHL": "PHI",  # Philadelphia (viejo código) -> Philadelphia 76ers
    "NJN": "BKN",  # New Jersey Nets -> Brooklyn Nets
    "CHH": "CHA",  # Charlotte Hornets (viejos) -> Charlotte Hornets actuales
    "VAN": "MEM",  # Vancouver Grizzlies -> Memphis Grizzlies
    "NOH": "NOP",  # New Orleans Hornets -> New Orleans Pelicans
    "NOK": "NOP",  # New Orleans/Oklahoma City Hornets -> Pelicans
}

teams_df = pd.DataFrame(nba_teams)
abbr_to_id = dict(zip(teams_df["abbreviation"], teams_df["id"]))

# Aplicar corrección directamente al dataframe de Lakers
df_lakers["OPONENTE"] = df_lakers["OPONENTE"].replace(abbr_fix_map)

# Volver a calcular oponentes únicos
opp_abbrevs = df_lakers["OPONENTE"].unique()
print("Oponentes únicos normalizados:", opp_abbrevs)

# Verificar si ya todos existen
missing = [abbr for abbr in opp_abbrevs if abbr not in abbr_to_id]
print("Abreviaturas fuera de catálogo después de normalizar:", missing)

if missing:
    print("Eliminando partidos contra equipos no-NBA:", missing)
    df_lakers = df_lakers[~df_lakers["OPONENTE"].isin(missing)].reset_index(drop=True)


Oponentes únicos normalizados: ['UTA' 'LAC' 'DAL' 'DEN' 'PHX' 'MIL' 'CLE' 'POR' 'OKC' 'CHI' 'GSW' 'NYK'
 'SAS' 'HOU' 'SAC' 'IND' 'BOS' 'WAS' 'ATL' 'PHI' 'BKN' 'DET' 'MIA' 'CHA'
 'ORL' 'MIN' 'MEM' 'TOR' 'NOP' 'BAR' 'MAC' 'SKT']
Abreviaturas fuera de catálogo después de normalizar: ['BAR', 'MAC', 'SKT']
Eliminando partidos contra equipos no-NBA: ['BAR', 'MAC', 'SKT']


In [48]:
# Esta vez pedimos TODOS los juegos donde el oponente fue LAL
opp_gamefinder = leaguegamefinder.LeagueGameFinder(
    vs_team_id_nullable=lakers_id   # <-- clave: vs_team_id, no team_id
)
df_opp_raw = opp_gamefinder.get_data_frames()[0]
print("df_opp_raw (todos los rivales vs LAL) shape:", df_opp_raw.shape)

# Nos quedamos solo con los GAME_ID que están en df_lakers (por si acaso)
df_opp_raw = df_opp_raw[df_opp_raw["GAME_ID"].isin(df_lakers["GAME_ID"])]
print("df_opp_raw filtrado shape:", df_opp_raw.shape)

# Columnas relevantes del rival (mismas métricas que Lakers)
opp_cols = [
    "GAME_ID",
    "WL",          # Win/Loss del rival
    "MIN",
    "PTS",
    "FGM", "FGA", "FG_PCT",
    "FG3M", "FG3A", "FG3_PCT",
    "FTM", "FTA", "FT_PCT",
    "OREB", "DREB", "REB",
    "AST", "STL", "BLK",
    "TOV", "PF",
]

df_opp = df_opp_raw[opp_cols].copy()

# Crear columna OPP_WIN (1 si el rival ganó, 0 si perdió)
df_opp["OPP_WIN"] = (df_opp["WL"] == "W").astype(int)
df_opp = df_opp.drop(columns=["WL"])

# Renombrar métricas del rival con prefijo OPP_
rename_map = {
    col: f"OPP_{col}" for col in df_opp.columns if col != "GAME_ID"
}
df_opp.rename(columns=rename_map, inplace=True)

print("df_opp shape (una fila por equipo-rival en cada juego):", df_opp.shape)
df_opp.head()


ReadTimeout: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)

In [49]:
df_opp.info()
df_opp.head()


<class 'pandas.core.frame.DataFrame'>
Index: 4083 entries, 0 to 4097
Data columns (total 21 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   GAME_ID      4083 non-null   object 
 1   OPP_MIN      4083 non-null   int64  
 2   OPP_PTS      4083 non-null   int64  
 3   OPP_FGM      4083 non-null   int64  
 4   OPP_FGA      4083 non-null   int64  
 5   OPP_FG_PCT   4083 non-null   float64
 6   OPP_FG3M     4083 non-null   int64  
 7   OPP_FG3A     4083 non-null   int64  
 8   OPP_FG3_PCT  4044 non-null   float64
 9   OPP_FTM      4083 non-null   int64  
 10  OPP_FTA      4083 non-null   int64  
 11  OPP_FT_PCT   4083 non-null   float64
 12  OPP_OREB     4083 non-null   int64  
 13  OPP_DREB     4083 non-null   int64  
 14  OPP_REB      4083 non-null   int64  
 15  OPP_AST      4083 non-null   int64  
 16  OPP_STL      4083 non-null   int64  
 17  OPP_BLK      4083 non-null   int64  
 18  OPP_TOV      4083 non-null   int64  
 19  OPP_PF     

,GAME_ID,OPP_MIN,OPP_PTS,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_STL,OPP_BLK,OPP_TOV,OPP_PF,OPP_OPP_WIN
0,0022501204,240,132,43,86,0.500,17,38,0.447,29,36,0.806,9,41,50,25,9,3,10,19,1
1,0022500362,240,108,36,102,0.353,13,36,0.361,23,26,0.885,17,37,54,21,5,4,6,17,0
2,0022500338,239,126,46,84,0.548,24,45,0.533,10,12,0.833,8,30,38,31,9,5,14,20,1
3,0022500336,240,120,44,89,0.494,15,37,0.405,17,21,0.810,14,27,41,39,9,8,15,20,0
4,0022500317,240,125,52,92,0.565,17,39,0.436,4,8,0.500,4,29,33,35,16,1,11,16,1


In [ ]:
# Dataframe combinado Lakers + Rival

df_lakers_opp = df_lakers.merge(
    df_opp,
    on="GAME_ID",
    how="left",
    validate="one_to_one"  # Si truena aquí es porque hay duplicados, se puede quitar el validate
)

print("df_lakers_opp shape:", df_lakers_opp.shape)
df_lakers_opp.head()

df_lakers_opp shape: (4083, 48)


,SEASON_ID,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,ES_CASA,OPONENTE,WIN,DAYS_REST,OPP_MIN,OPP_PTS,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_STL,OPP_BLK,OPP_TOV,OPP_PF,OPP_OPP_WIN
0,21983,0028300010,1983-10-29,LAL @ UTH,W,240,120,46,98,0.469,0,1,0.000,28,39,0.718,29,31,60,25,10,10,20,41,0,UTA,1,2,240,115,35,77,0.455,1,3,0.333,44,60,0.733,15,26,41,21,8,10,17,27,0
1,21983,0028300035,1983-11-02,LAL @ SDC,L,240,106,43,96,0.448,1,2,0.500,19,25,0.760,16,33,49,29,14,7,24,24,0,LAC,0,4,240,110,47,91,0.516,0,1,0.000,16,23,0.696,11,31,42,30,12,8,20,24,1
2,21983,0028300053,1983-11-05,LAL @ DAL,L,240,102,47,97,0.485,0,2,0.000,8,12,0.667,18,23,41,29,7,8,20,27,0,DAL,0,3,240,107,43,97,0.443,0,2,0.000,21,26,0.808,22,26,48,26,9,4,16,20,1
3,21983,0028300066,1983-11-08,LAL @ DEN,W,240,133,49,106,0.462,2,3,0.667,33,36,0.917,22,31,53,30,11,8,23,31,0,DEN,1,3,240,124,44,96,0.458,0,8,0.000,36,42,0.857,14,30,44,28,15,4,24,26,0
4,21983,0028300068,1983-11-09,LAL vs. DAL,W,240,120,50,94,0.532,2,3,0.667,18,28,0.643,16,33,49,33,7,6,17,31,1,DAL,1,1,240,106,34,83,0.410,0,2,0.000,38,46,0.826,13,30,43,22,7,2,18,23,0


### 2.2.2 Definir columnas de features y target y Creación de Datasets de Entrenamiento y Validación

In [50]:
feature_cols_22 = [
    # Features de los Lakers (igual que en 2.1)
    'ES_CASA',
    'WIN',
    'MIN',
    'FGM', 'FGA', 'FG_PCT',
    'FG3M', 'FG3A', 'FG3_PCT',
    'FTM', 'FTA', 'FT_PCT',
    'OREB', 'DREB', 'REB',
    'AST', 'STL', 'BLK',
    'TOV', 'PF',
    'DAYS_REST',
    
    # Features del rival (prefijo OPP_)
    'OPP_MIN',
    'OPP_PTS',
    'OPP_FGM', 'OPP_FGA', 'OPP_FG_PCT',
    'OPP_FG3M', 'OPP_FG3A', 'OPP_FG3_PCT',
    'OPP_FTM', 'OPP_FTA', 'OPP_FT_PCT',
    'OPP_OREB', 'OPP_DREB', 'OPP_REB',
    'OPP_AST', 'OPP_STL', 'OPP_BLK',
    'OPP_TOV', 'OPP_PF'
]

target_col_22 = 'PTS' 

data_features_22 = df_lakers_opp[feature_cols_22].values.astype('float32')
data_target_22 = df_lakers_opp[target_col_22].values.astype('float32')

print("Features 2.2 shape (sin secuencias):", data_features_22.shape)
print("Target 2.2 shape (sin secuencias):", data_target_22.shape)


Features 2.2 shape (sin secuencias): (4083, 40)
Target 2.2 shape (sin secuencias): (4083,)


### 2.2.3 Preparación de datos secuenciales para el nuevo modelo

In [51]:
def prepare_data_for_seq_len_22(seq_len):
    X, y = build_sequences(data_features_22, data_target_22, seq_len=seq_len)
    print(f"\n[2.2] SEQ_LEN = {seq_len} -> X: {X.shape}, y: {y.shape}")
    
    n = X.shape[0]
    train_size = int(n * 0.7)
    val_size   = int(n * 0.15)
    
    X_train = X[:train_size]
    y_train = y[:train_size]
    X_val   = X[train_size:train_size+val_size]
    y_val   = y[train_size:train_size+val_size]
    X_test  = X[train_size+val_size:]
    y_test  = y[train_size+val_size:]
    
    num_features = X.shape[2]
    
    # Escalado (igual que en 2.1)
    X_train_2d = X_train.reshape(-1, num_features)
    X_val_2d   = X_val.reshape(-1, num_features)
    X_test_2d  = X_test.reshape(-1, num_features)
    
    scaler = StandardScaler()
    X_train_scaled_2d = scaler.fit_transform(X_train_2d)
    X_val_scaled_2d   = scaler.transform(X_val_2d)
    X_test_scaled_2d  = scaler.transform(X_test_2d)
    
    X_train_scaled = X_train_scaled_2d.reshape(X_train.shape)
    X_val_scaled   = X_val_scaled_2d.reshape(X_val.shape)
    X_test_scaled  = X_test_scaled_2d.reshape(X_test.shape)
    
    return (
        X_train_scaled, y_train,
        X_val_scaled, y_val,
        X_test_scaled, y_test,
        num_features, scaler
    )


### 2.2.4 Flujo de Keras Tuner y MLflow

In [26]:
mlflow.set_experiment("nba_lakers_lstm_con_rival")  # nuevo experimento para Parte 2.2

seq_len_candidates_22 = [3, 5, 10]
max_epochs_22 = 40

results_22 = []

for seq_len in seq_len_candidates_22:
    (
        X_train_scaled, y_train,
        X_val_scaled, y_val,
        X_test_scaled, y_test,
        num_features, scaler
    ) = prepare_data_for_seq_len_22(seq_len)

    with mlflow.start_run(run_name=f"tuning_seq{seq_len}_rival"):

        mlflow.log_param("seq_len", seq_len)

        def hypermodel(hp):
            return build_lstm_model(hp, seq_len, num_features)

        tuner = BatchSizeTuner(
            hypermodel,
            objective='val_loss',
            max_epochs=max_epochs_22,
            factor=3,
            directory=base_dir,
            project_name=f"seq_{seq_len}_rival",
            overwrite=True
        )

        stop_early = EarlyStopping(
            monitor='val_loss',
            patience=5,
            restore_best_weights=True
        )

        tuner.search(
            X_train_scaled, y_train,
            validation_data=(X_val_scaled, y_val),
            callbacks=[stop_early],
            verbose=1
        )

        # Mejor conjunto de hiperparámetros
        best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]
        best_trial = tuner.oracle.get_best_trials(num_trials=1)[0]
        best_val_loss = best_trial.score

        # Construir modelo ganador y reentrenar con train+val
        best_model = tuner.hypermodel.build(best_hp)

        history = best_model.fit(
            np.concatenate([X_train_scaled, X_val_scaled], axis=0),
            np.concatenate([y_train, y_val], axis=0),
            epochs=max_epochs_22,
            batch_size=best_hp.get('batch_size'),
            callbacks=[stop_early],
            verbose=0
        )

        test_loss, test_mae = best_model.evaluate(X_test_scaled, y_test, verbose=0)

        # Log en MLflow
        mlflow.log_metric("best_val_loss", float(best_val_loss))
        mlflow.log_metric("test_mse", float(test_loss))
        mlflow.log_metric("test_mae", float(test_mae))

        for name, value in best_hp.values.items():
            mlflow.log_param(f"best_{name}", value)

        run_result = {
            "seq_len": seq_len,
            "best_val_loss": float(best_val_loss),
            "test_mse": float(test_loss),
            "test_mae": float(test_mae),
        }
        for name, value in best_hp.values.items():
            run_result[name] = value

        results_22.append(run_result)

Trial 90 Complete [00h 00m 18s]
val_loss: 8832.857421875

Best val_loss So Far: 187.379638671875
Total elapsed time: 00h 15m 02s


C:\Users\luis_\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\keras\src\callbacks\early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss,mae
  current = self.get_monitor_value(logs)


### 2.2.5 Métricas y Evaluación

In [27]:
print_results_for_seq_len(results_22, 3)


======================= RESULTADOS COMPLETOS: SEQ_LEN = 3 =======================

best_val_loss : 187.28448486328125
test_mse       : 252.99362182617188
test_mae       : 12.887859344482422

Hiperparámetros ganadores:
  - lstm_units: 112
  - dropout_rate: 0.30000000000000004
  - dense_units: 16
  - optimizer: sgd
  - learning_rate: 0.0003383182105335443
  - clipnorm: 0.0
  - momentum: 0.6
  - batch_size: 32
  - tuner/epochs: 40
  - tuner/initial_epoch: 0
  - tuner/bracket: 0
  - tuner/round: 0



In [28]:
print_results_for_seq_len(results_22, 5)


======================= RESULTADOS COMPLETOS: SEQ_LEN = 5 =======================

best_val_loss : 187.4408721923828
test_mse       : 252.73110961914062
test_mae       : 12.881135940551758

Hiperparámetros ganadores:
  - lstm_units: 32
  - dropout_rate: 0.2
  - dense_units: 8
  - optimizer: sgd
  - learning_rate: 0.0012306883948819032
  - clipnorm: 0.0
  - momentum: 0.8999999999999999
  - batch_size: 64
  - tuner/epochs: 5
  - tuner/initial_epoch: 2
  - tuner/bracket: 3
  - tuner/round: 1
  - tuner/trial_id: 0017



In [29]:
print_results_for_seq_len(results_22, 10)


======================= RESULTADOS COMPLETOS: SEQ_LEN = 10 =======================

best_val_loss : 187.379638671875
test_mse       : 253.57949829101562
test_mae       : 12.90460205078125

Hiperparámetros ganadores:
  - lstm_units: 16
  - dropout_rate: 0.4
  - dense_units: 64
  - optimizer: sgd
  - learning_rate: 0.0005986605913831514
  - clipnorm: 0.0
  - momentum: 0.3
  - batch_size: 32
  - tuner/epochs: 40
  - tuner/initial_epoch: 14
  - tuner/bracket: 2
  - tuner/round: 2
  - tuner/trial_id: 0067



In [30]:
y_pred_22 = best_model.predict(X_test_scaled).flatten()
r2_22 = r2_score(y_test, y_pred_22)
print("R² (modelo con stats del rival):", r2_22)

20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
R² (modelo con stats del rival): -0.3034573793411255


In [31]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Entrenamiento
y_train_pred = best_model.predict(X_train_scaled).flatten()
train_mae = mean_absolute_error(y_train, y_train_pred)
train_mse = mean_squared_error(y_train, y_train_pred)

# Validación+test (según quieras, aquí test)
y_test_pred = best_model.predict(X_test_scaled).flatten()
test_mae = mean_absolute_error(y_test, y_test_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
test_r2  = r2_score(y_test, y_test_pred)

print("MAE train :", train_mae)
print("MSE train :", train_mse)
print("MAE test  :", test_mae)
print("MSE test  :", test_mse)
print("R² test   :", test_r2)



90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
MAE train : 10.787086486816406
MSE train : 185.3451385498047
MAE test  : 12.904601097106934
MSE test  : 253.57949829101562
R² test   : -0.3034573793411255


In [33]:
y_mean = np.mean(y_test)
baseline_pred = np.full_like(y_test, y_mean)

baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_mse = mean_squared_error(y_test, baseline_pred)
baseline_r2  = r2_score(y_test, baseline_pred)

print("Baseline MAE:", baseline_mae)
print("Baseline MSE:", baseline_mse)
print("Baseline R² :", baseline_r2)


Baseline MAE: 10.986564636230469
Baseline MSE: 194.54376220703125
Baseline R² : 0.0


## Red Neuronal Densa (ChatGPT)

In [56]:
# Partimos de df_lakers_opp ya construido
df = df_lakers_opp.copy()

# Aseguramos orden temporal
df = df.sort_values("GAME_DATE").reset_index(drop=True)

# -----------------------------
# Rolling averages (últimos 5)
# -----------------------------
window = 5

lakers_base_cols = ["PTS", "FG_PCT", "FG3_PCT", "REB", "AST"]
opp_base_cols    = ["OPP_PTS", "OPP_FG_PCT", "OPP_FG3_PCT", "OPP_REB", "OPP_AST"]

for col in lakers_base_cols:
    df[f"LAL_{col}_roll{window}"] = (
        df[col].rolling(window=window, min_periods=window).mean().shift(1)
    )

for col in opp_base_cols:
    df[f"{col}_roll{window}"] = (
        df[col].rolling(window=window, min_periods=window).mean().shift(1)
    )

# -----------------------------
# Rachas de victorias (streaks)
# -----------------------------
def compute_streak(series_win):
    streaks = []
    streak = 0
    for w in series_win:
        if w == 1:
            streak += 1
        else:
            streak = 0
        streaks.append(streak)
    # shift(1) para que la racha sea "antes del partido actual"
    return pd.Series(streaks).shift(1)

# Asegúrate de que WIN y OPP_WIN son 0/1
df["LAL_WIN_STREAK"] = compute_streak(df["WIN"])
df["OPP_WIN_STREAK"] = compute_streak(df["OPP_OPP_WIN"])

# -----------------------------
# Back-to-back (descanso 0 días)
# -----------------------------
df["LAL_B2B"] = (df["DAYS_REST"] == 0).astype(int)

# -----------------------------
# Definir columnas de features y target
# -----------------------------
feature_cols_b = [
    # contexto básico
    "ES_CASA",
    "DAYS_REST",
    "LAL_B2B",
    
    # forma reciente Lakers
    f"LAL_PTS_roll{window}",
    f"LAL_FG_PCT_roll{window}",
    f"LAL_FG3_PCT_roll{window}",
    f"LAL_REB_roll{window}",
    f"LAL_AST_roll{window}",
    "LAL_WIN_STREAK",
    
    # forma reciente rival
    f"OPP_PTS_roll{window}",
    f"OPP_FG_PCT_roll{window}",
    f"OPP_FG3_PCT_roll{window}",
    f"OPP_REB_roll{window}",
    f"OPP_AST_roll{window}",
    "OPP_WIN_STREAK",
]

target_col_b = "PTS"  # puntos de Lakers en ese partido

# -----------------------------
# Aquí se crea df_model (la parte que te faltaba)
# -----------------------------
df_model = df.dropna(subset=feature_cols_b + [target_col_b]).reset_index(drop=True)

print("Shape df_model:", df_model.shape)
print("Primeras columnas de df_model:", df_model.columns.tolist())


Shape df_model: (3917, 61)
Primeras columnas de df_model: ['SEASON_ID', 'GAME_ID', 'GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'PTS', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'ES_CASA', 'OPONENTE', 'WIN', 'DAYS_REST', 'OPP_MIN', 'OPP_PTS', 'OPP_FGM', 'OPP_FGA', 'OPP_FG_PCT', 'OPP_FG3M', 'OPP_FG3A', 'OPP_FG3_PCT', 'OPP_FTM', 'OPP_FTA', 'OPP_FT_PCT', 'OPP_OREB', 'OPP_DREB', 'OPP_REB', 'OPP_AST', 'OPP_STL', 'OPP_BLK', 'OPP_TOV', 'OPP_PF', 'OPP_OPP_WIN', 'LAL_PTS_roll5', 'LAL_FG_PCT_roll5', 'LAL_FG3_PCT_roll5', 'LAL_REB_roll5', 'LAL_AST_roll5', 'OPP_PTS_roll5', 'OPP_FG_PCT_roll5', 'OPP_FG3_PCT_roll5', 'OPP_REB_roll5', 'OPP_AST_roll5', 'LAL_WIN_STREAK', 'OPP_WIN_STREAK', 'LAL_B2B']


In [57]:
# Ya está ordenado por fecha
n = len(df_model)
train_end = int(n * 0.7)
val_end   = int(n * 0.85)

train_df = df_model.iloc[:train_end]
val_df   = df_model.iloc[train_end:val_end]
test_df  = df_model.iloc[val_end:]

X_train = train_df[feature_cols_b].values.astype("float32")
y_train = train_df[target_col_b].values.astype("float32")

X_val   = val_df[feature_cols_b].values.astype("float32")
y_val   = val_df[target_col_b].values.astype("float32")

X_test  = test_df[feature_cols_b].values.astype("float32")
y_test  = test_df[target_col_b].values.astype("float32")

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)


Train: (2741, 15) Val: (588, 15) Test: (588, 15)


In [58]:
from sklearn.preprocessing import StandardScaler

scaler_b = StandardScaler()
X_train_scaled = scaler_b.fit_transform(X_train)
X_val_scaled   = scaler_b.transform(X_val)
X_test_scaled  = scaler_b.transform(X_test)


In [59]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

def build_mlp_model(input_dim):
    model = tf.keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(1)  # regresión
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="mse",
        metrics=["mae"]
    )
    return model

input_dim = X_train_scaled.shape[1]
mlp_model = build_mlp_model(input_dim)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = mlp_model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=200,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)


Epoch 1/200
86/86 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 10058.0605 - mae: 99.3362 - val_loss: 8062.0215 - val_mae: 88.5692
Epoch 2/200
86/86 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 5487.6089 - mae: 70.3306 - val_loss: 1795.4624 - val_mae: 37.0665
Epoch 3/200
86/86 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1613.0992 - mae: 33.7391 - val_loss: 924.1995 - val_mae: 24.3680
Epoch 4/200
86/86 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1144.9077 - mae: 27.3543 - val_loss: 651.2697 - val_mae: 20.4190
Epoch 5/200
86/86 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 914.0450 - mae: 24.4025 - val_loss: 566.6995 - val_mae: 18.6874
Epoch 6/200
86/86 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 818.3356 - mae: 22.7331 - val_loss: 442.8829 - val_mae: 16.6680
Epoch 7/200
86/86 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 747.4603 - mae: 21.7674 - val_loss: 397.0466 - val_mae: 15.8324
Epoch 8/200
86/86 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 724.7413 - mae: 21.1555 - val_loss: 347.3976 - val_mae: 14.8124
Epoch 9/2

In [60]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Evaluación directa de Keras
test_loss, test_mae = mlp_model.evaluate(X_test_scaled, y_test, verbose=0)
print("MLP - test MSE (Keras):", test_loss)
print("MLP - test MAE (Keras):", test_mae)

# Métricas más completas
y_pred = mlp_model.predict(X_test_scaled).flatten()

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)

print("\nMLP - MAE:", mae)
print("MLP - MSE:", mse)
print("MLP - R² :", r2)

# Baseline media para comparar
y_mean = np.mean(y_test)
baseline_pred = np.full_like(y_test, y_mean)

baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_mse = mean_squared_error(y_test, baseline_pred)
baseline_r2  = r2_score(y_test, baseline_pred)

print("\nBaseline MAE:", baseline_mae)
print("Baseline MSE:", baseline_mse)
print("Baseline R² :", baseline_r2)


MLP - test MSE (Keras): 203.68870544433594
MLP - test MAE (Keras): 11.384740829467773
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step

MLP - MAE: 11.384740829467773
MLP - MSE: 203.68869018554688
MLP - R² : -0.03834092617034912

Baseline MAE: 11.024249076843262
Baseline MSE: 196.16746520996094
Baseline R² : 0.0


In [ ]:
import mlflow
import mlflow.keras  # o mlflow.tensorflow según tu versión

mlflow.set_experiment("nba_lakers_mlp_tabular")

with mlflow.start_run(run_name="mlp_tabular_roll5"):
    mlflow.log_param("model_type", "MLP_tabular")
    mlflow.log_param("window", window)
    mlflow.log_param("hidden_layers", [64, 32])
    mlflow.log_param("dropout", 0.3)
    mlflow.log_param("optimizer", "adam")
    mlflow.log_param("learning_rate", 1e-3)
    mlflow.log_metric("test_mae", float(mae))
    mlflow.log_metric("test_mse", float(mse))
    mlflow.log_metric("test_r2", float(r2))
    
    # Guardar el modelo
    mlflow.keras.log_model(mlp_model, artifact_path="mlp_model")


In [61]:
df_lakers_opp.head()

,SEASON_ID,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,ES_CASA,OPONENTE,WIN,DAYS_REST,OPP_MIN,OPP_PTS,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_STL,OPP_BLK,OPP_TOV,OPP_PF,OPP_OPP_WIN
0,21983,0028300010,1983-10-29,LAL @ UTH,W,240,120,46,98,0.469,0,1,0.000,28,39,0.718,29,31,60,25,10,10,20,41,0,UTA,1,2,240,115,35,77,0.455,1,3,0.333,44,60,0.733,15,26,41,21,8,10,17,27,0
1,21983,0028300035,1983-11-02,LAL @ SDC,L,240,106,43,96,0.448,1,2,0.500,19,25,0.760,16,33,49,29,14,7,24,24,0,LAC,0,4,240,110,47,91,0.516,0,1,0.000,16,23,0.696,11,31,42,30,12,8,20,24,1
2,21983,0028300053,1983-11-05,LAL @ DAL,L,240,102,47,97,0.485,0,2,0.000,8,12,0.667,18,23,41,29,7,8,20,27,0,DAL,0,3,240,107,43,97,0.443,0,2,0.000,21,26,0.808,22,26,48,26,9,4,16,20,1
3,21983,0028300066,1983-11-08,LAL @ DEN,W,240,133,49,106,0.462,2,3,0.667,33,36,0.917,22,31,53,30,11,8,23,31,0,DEN,1,3,240,124,44,96,0.458,0,8,0.000,36,42,0.857,14,30,44,28,15,4,24,26,0
4,21983,0028300068,1983-11-09,LAL vs. DAL,W,240,120,50,94,0.532,2,3,0.667,18,28,0.643,16,33,49,33,7,6,17,31,1,DAL,1,1,240,106,34,83,0.410,0,2,0.000,38,46,0.826,13,30,43,22,7,2,18,23,0


## Nuevo engeneering de features

In [62]:
import numpy as np
import pandas as pd

df = df_lakers_opp.copy()

# Asegurar orden temporal
df = df.sort_values("GAME_DATE").reset_index(drop=True)

# -----------------------------
# Posesiones estimadas Lakers y rival
# -----------------------------
# Ajusta nombres si en tu df son distintos
lal_poss = (
    df["FGA"]
    + 0.44 * df["FTA"]
    - df["OREB"]
    + df["TOV"]
)

opp_poss = (
    df["OPP_FGA"]
    + 0.44 * df["OPP_FTA"]
    - df["OPP_OREB"]
    + df["OPP_TOV"]
)

# Posesiones del partido ≈ media de las dos estimaciones
df["POSS_EST"] = (lal_poss + opp_poss) / 2

# Evitar divisiones raras
df = df[df["POSS_EST"] > 0].reset_index(drop=True)

# -----------------------------
# Ratings avanzados por partido
# -----------------------------
df["LAL_OFF_RTG"] = 100 * df["PTS"] / df["POSS_EST"]
df["LAL_DEF_RTG"] = 100 * df["OPP_PTS"] / df["POSS_EST"]
df["LAL_NET_RTG"] = df["LAL_OFF_RTG"] - df["LAL_DEF_RTG"]

df["OPP_OFF_RTG"] = 100 * df["OPP_PTS"] / df["POSS_EST"]
df["OPP_DEF_RTG"] = 100 * df["PTS"] / df["POSS_EST"]
df["OPP_NET_RTG"] = df["OPP_OFF_RTG"] - df["OPP_DEF_RTG"]

df[["GAME_DATE", "PTS", "OPP_PTS", "LAL_OFF_RTG", "LAL_DEF_RTG", "OPP_OFF_RTG", "OPP_DEF_RTG"]].head()


,GAME_DATE,PTS,OPP_PTS,LAL_OFF_RTG,LAL_DEF_RTG,OPP_OFF_RTG,OPP_DEF_RTG
0,1983-10-29,120,115,113.442995,108.716203,108.716203,113.442995
1,1983-11-02,106,110,94.171997,97.725657,97.725657,94.171997
2,1983-11-05,102,107,98.684211,103.521672,103.521672,98.684211
3,1983-11-08,133,124,107.552968,100.274947,100.274947,107.552968
4,1983-11-09,120,106,111.337911,98.348488,98.348488,111.337911


In [63]:
window = 5  # partidos recientes

# Stats base Lakers / rival
lakers_base_cols = ["PTS", "FG_PCT", "FG3_PCT", "REB", "AST"]
opp_base_cols    = ["OPP_PTS", "OPP_FG_PCT", "OPP_FG3_PCT", "OPP_REB", "OPP_AST"]

# Rolling de stats simples
for col in lakers_base_cols:
    df[f"LAL_{col}_roll{window}"] = (
        df[col].rolling(window=window, min_periods=window).mean().shift(1)
    )

for col in opp_base_cols:
    df[f"{col}_roll{window}"] = (
        df[col].rolling(window=window, min_periods=window).mean().shift(1)
    )

# Rolling de ratings avanzados
rating_cols_lal = ["LAL_OFF_RTG", "LAL_DEF_RTG", "LAL_NET_RTG"]
rating_cols_opp = ["OPP_OFF_RTG", "OPP_DEF_RTG", "OPP_NET_RTG"]

for col in rating_cols_lal:
    df[f"{col}_roll{window}"] = (
        df[col].rolling(window=window, min_periods=window).mean().shift(1)
    )

for col in rating_cols_opp:
    df[f"{col}_roll{window}"] = (
        df[col].rolling(window=window, min_periods=window).mean().shift(1)
    )


In [65]:
def compute_streak(series_win):
    streaks = []
    streak = 0
    for w in series_win:
        if w == 1:
            streak += 1
        else:
            streak = 0
        streaks.append(streak)
    # shift(1): racha *antes* del partido
    return pd.Series(streaks).shift(1)

# Asegúrate de que WIN y OPP_WIN sean 0/1
df["LAL_WIN_STREAK"] = compute_streak(df["WIN"])
df["OPP_WIN_STREAK"] = compute_streak(df["OPP_OPP_WIN"])


In [66]:
# B2B si DAYS_REST == 0
df["LAL_B2B"] = (df["DAYS_REST"] == 0).astype(int)

# Si no tienes DAYS_REST del rival, puedes omitir OPP_B2B
# Si lo tuvieras como OPP_DAYS_REST:
# df["OPP_B2B"] = (df["OPP_DAYS_REST"] == 0).astype(int)


In [67]:
feature_cols_b = [
    # Contexto básico
    "ES_CASA",
    "DAYS_REST",
    "LAL_B2B",
    
    # Forma reciente Lakers (rolling stats)
    f"LAL_PTS_roll{window}",
    f"LAL_FG_PCT_roll{window}",
    f"LAL_FG3_PCT_roll{window}",
    f"LAL_REB_roll{window}",
    f"LAL_AST_roll{window}",
    "LAL_WIN_STREAK",
    
    # Forma reciente rival
    f"OPP_PTS_roll{window}",
    f"OPP_FG_PCT_roll{window}",
    f"OPP_FG3_PCT_roll{window}",
    f"OPP_REB_roll{window}",
    f"OPP_AST_roll{window}",
    "OPP_WIN_STREAK",
    
    # Ratings avanzados (forma reciente, no partido puntual)
    f"LAL_OFF_RTG_roll{window}",
    f"LAL_DEF_RTG_roll{window}",
    f"LAL_NET_RTG_roll{window}",
    f"OPP_OFF_RTG_roll{window}",
    f"OPP_DEF_RTG_roll{window}",
    f"OPP_NET_RTG_roll{window}",
]

target_col_b = "PTS"

# Quitamos filas donde falte algo (por los rollings/streaks)
df_model = df.dropna(subset=feature_cols_b + [target_col_b]).reset_index(drop=True)

print("Shape df_model_ext:", df_model.shape)


Shape df_model_ext: (3917, 74)


In [68]:
from sklearn.preprocessing import StandardScaler

# Split temporal 70 / 15 / 15
n = len(df_model)
train_end = int(n * 0.7)
val_end   = int(n * 0.85)

train_df = df_model.iloc[:train_end]
val_df   = df_model.iloc[train_end:val_end]
test_df  = df_model.iloc[val_end:]

X_train = train_df[feature_cols_b].values.astype("float32")
y_train = train_df[target_col_b].values.astype("float32")

X_val   = val_df[feature_cols_b].values.astype("float32")
y_val   = val_df[target_col_b].values.astype("float32")

X_test  = test_df[feature_cols_b].values.astype("float32")
y_test  = test_df[target_col_b].values.astype("float32")

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

# Escalado
scaler_b = StandardScaler()
X_train_scaled = scaler_b.fit_transform(X_train)
X_val_scaled   = scaler_b.transform(X_val)
X_test_scaled  = scaler_b.transform(X_test)


Train: (2741, 21) Val: (588, 21) Test: (588, 21)


In [69]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

def build_mlp_model(input_dim):
    model = tf.keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(1)  # regresión
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
        loss="mse",
        metrics=["mae"]
    )
    return model

input_dim = X_train_scaled.shape[1]
mlp_model = build_mlp_model(input_dim)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=12,
    restore_best_weights=True
)

history = mlp_model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=200,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)


Epoch 1/200
86/86 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - loss: 10656.2744 - mae: 102.3566 - val_loss: 9977.4512 - val_mae: 98.8975
Epoch 2/200
86/86 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 9510.4521 - mae: 96.5571 - val_loss: 7875.4551 - val_mae: 87.3328
Epoch 3/200
86/86 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 7039.9009 - mae: 82.0831 - val_loss: 4465.3325 - val_mae: 62.7945
Epoch 4/200
86/86 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 4007.6833 - mae: 58.8469 - val_loss: 2068.6145 - val_mae: 39.0501
Epoch 5/200
86/86 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2209.3936 - mae: 40.8473 - val_loss: 1710.4573 - val_mae: 34.6649
Epoch 6/200
86/86 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1810.7462 - mae: 35.5022 - val_loss: 1626.9677 - val_mae: 33.2366
Epoch 7/200
86/86 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1659.4417 - mae: 33.6579 - val_loss: 1477.2283 - val_mae: 31.5790
Epoch 8/200
86/86 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1475.9285 - mae: 31.5497 - val_loss: 1324.9570 - val_mae: 29.689

In [70]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Evaluación Keras
test_loss, test_mae = mlp_model.evaluate(X_test_scaled, y_test, verbose=0)
print("MLP - test MSE (Keras):", test_loss)
print("MLP - test MAE (Keras):", test_mae)

# Métricas detalladas
y_pred = mlp_model.predict(X_test_scaled).flatten()

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)

print("\nMLP avanzado - MAE:", mae)
print("MLP avanzado - MSE:", mse)
print("MLP avanzado - R² :", r2)

# Baseline media
y_mean = np.mean(y_test)
baseline_pred = np.full_like(y_test, y_mean)

baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_mse = mean_squared_error(y_test, baseline_pred)
baseline_r2  = r2_score(y_test, baseline_pred)

print("\nBaseline MAE:", baseline_mae)
print("Baseline MSE:", baseline_mse)
print("Baseline R² :", baseline_r2)


MLP - test MSE (Keras): 206.6910400390625
MLP - test MAE (Keras): 11.507651329040527
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step

MLP avanzado - MAE: 11.507651329040527
MLP avanzado - MSE: 206.69100952148438
MLP avanzado - R² : -0.05364573001861572

Baseline MAE: 11.024249076843262
Baseline MSE: 196.16746520996094
Baseline R² : 0.0
